&nbsp;

# `salmon quant` result parse notebook

- `salmon` によって 発現定量を行った RNA-sequencing (RNA-seq) の ライブラリーの推定や、定量結果のメタデータをデータフレームとして parse するための notebook
- [salmon の LIBTYPE](https://salmon.readthedocs.io/en/latest/salmon.html#what-s-this-libtype) の判定結果が記載してある `lib_format_counts.json` をデータフレームとしてまとめていく
- About `lib_format_counts.json` -> [Salmon lib_format_counts.json - explanation](https://www.biostars.org/p/9556263/)

In [1]:
import os
import pathlib

import plotly.express as px
import polars as pl
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Metadata

In [2]:
rnaseq_all_metadata = pl.read_csv(
    "../../Data/metadata/metadata_2610/rna_seq_all_metadata_simple_261001.tsv",
    separator="\t",
    ignore_errors=True
)

pl.Config.set_tbl_rows(2)
display(rnaseq_all_metadata)

sample_heat_id,sample_control_id,sample_name,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,sample,stress treatment temperature,stress treatment time,control treatment temperature,rna_library,normalized_label,item_type,reference_doi,strandness,strandness_evidence
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR27681984""","""SRR27681986""","""wi-38-htert_40°c_rna-seq_rep1 vs wi-38-htert_37°c_rna-seq_rep1""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 37°C, RNA-seq, rep1""","""SRX23349232 vs SRX23349230""","""GSM8029688_r1 vs GSM8029686_r1""","""SRS20210616 vs SRS20210614""","""PRJNA1067599""","""2024-10-16""","""GSE253874""","""ILLUMINA""","""Illumina HiSeq 2000""","""TRANSCRIPTOMIC""","""SINGLE""","""total RNA""","""WI-38-hTERT""","""40 C""","""120 min.""","""37 C""","""TruSeq Stranded Total RNA Library Prep Kit""","""TruSeq Stranded Total RNA Library Prep Kit""","""library_preparation""","""https://doi.org/10.1111/gtc.13140""","""stranded""","""https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-refer…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HRR255914""","""HRR255912""","""RNAseq_Neuron_39_rep2 vs RNAseq_Neuron_37_rep2""","""RNAseq_Neuron_39_rep2 vs RNAseq_Neuron_37_rep2""","""HRX214101 vs HRX214099""","""RNAseq_Neuron_39 vs RNAseq_Neuron_37""","""HRS192117 vs HRS192116""","""PRJCA005446""","""2019-10-30""","""PRJCA005446""","""ILLUMINA""","""Illumina HiSeq X Ten""","""GENOMIC""","""PAIRED""","""total RNA""","""human neuron (hNeuron)""","""39 C""","""2880 min.""","""37 C""","""NEBNext® UltraTM Directional RNA Library Prep Kit""","""NEBNext Ultra Directional RNA Library Prep Kit for Illumina""","""library_preparation""","""https://doi.org/10.1007/s13238-021-00887-y""","""stranded""","""https://www.neb.com/en/products/e7420-nebnext-ultra-directional-rna-library-prep-kit-for-illumina"""


&nbsp;

&nbsp;

## ALL Metadata Parse Process

- メタデータの `sample_heat_id` と `sample_control_id` から unique な sample ID を取り出す
- `out/salmon_gencode_result` 配下の `{sample_id}` / `{sample_id}_1` / `{sample_id}_2` ディレクトリにある `lib_format_counts.json` を探す
- PE 定量は mate-1 FASTQ 名に合わせて `_1` が付くことが多く、一部サンプルは `_2` になる
- 各 JSON を読み込み、Polars DataFrame として縦方向に concat する

In [3]:
# Unique sample IDs from heat / control columns (one row per accession)
unique_sample_ids = (
    rnaseq_all_metadata
    .select("sample_heat_id", "sample_control_id")
    .unpivot(
        on=["sample_heat_id", "sample_control_id"],
        variable_name="sample_role",
        value_name="sample_id",
    )
    .drop_nulls("sample_id")
    .select("sample_id")
    .unique(maintain_order=True)
)

pl.Config.set_tbl_rows(2)
print(f"unique sample IDs: {unique_sample_ids.height}")
display(unique_sample_ids)

unique sample IDs: 744


sample_id
str
"""SRR27681984"""
…
"""HRR255912"""


In [4]:
# Index lib_format_counts.json paths, then map directory names to metadata sample_id.
# Prefer an exact directory match; otherwise accept "{sample_id}_1" or "{sample_id}_2"
# (PE quant often uses mate FASTQ names; a few SE samples use _2).
SALMON_RESULT_ROOT = pathlib.Path("../../out/salmon_gencode_result")
json_paths = sorted(SALMON_RESULT_ROOT.glob("*/*/lib_format_counts.json"))

json_index = pl.DataFrame(
    {
        "lib_format_json_path": [str(p) for p in json_paths],
        "quant_dir_name": [p.parent.name for p in json_paths],
        "quant_layout": [p.parent.parent.name for p in json_paths],
    }
)

json_by_sample_id = pl.concat(
    [
        json_index.with_columns(pl.col("quant_dir_name").alias("sample_id")),
        json_index.filter(pl.col("quant_dir_name").str.ends_with("_1")).with_columns(
            pl.col("quant_dir_name").str.strip_suffix("_1").alias("sample_id")
        ),
        json_index.filter(pl.col("quant_dir_name").str.ends_with("_2")).with_columns(
            pl.col("quant_dir_name").str.strip_suffix("_2").alias("sample_id")
        ),
    ],
    how="vertical",
).unique(subset="sample_id", keep="first", maintain_order=True)

sample_json = unique_sample_ids.join(
    json_by_sample_id,
    on="sample_id",
    how="left",
    maintain_order="left",
)

found_lib_format = sample_json.filter(pl.col("lib_format_json_path").is_not_null())
missing_lib_format = sample_json.filter(pl.col("lib_format_json_path").is_null())

print(f"json files under salmon_gencode_result: {json_index.height}")
print(f"matched to metadata: {found_lib_format.height}")
print(f"metadata IDs without json: {missing_lib_format.height}")
display(found_lib_format)
display(missing_lib_format)

json files under salmon_gencode_result: 900
matched to metadata: 744
metadata IDs without json: 0


sample_id,lib_format_json_path,quant_dir_name,quant_layout
str,str,str,str
"""SRR27681984""","""../../out/salmon_gencode_result/HS_human_salmon_quant_se_gencode/SRR27681984/lib_format_counts.json""","""SRR27681984""","""HS_human_salmon_quant_se_gencode"""
…,…,…,…
"""HRR255912""","""../../out/salmon_gencode_result/HS_human_salmon_quant_pe_gencode/HRR255912_1/lib_format_counts.json""","""HRR255912_1""","""HS_human_salmon_quant_pe_gencode"""


sample_id,lib_format_json_path,quant_dir_name,quant_layout
str,str,str,str


In [5]:
# Read each matched JSON (one row per file) and concatenate
lib_format_counts = pl.concat(
    [
        pl.read_json(path).with_columns(
            pl.lit(sample_id).alias("sample_id"),
            pl.lit(quant_dir_name).alias("quant_dir_name"),
            pl.lit(quant_layout).alias("quant_layout"),
            pl.lit(path).alias("lib_format_json_path"),
        )
        for sample_id, quant_dir_name, quant_layout, path in found_lib_format.select(
            "sample_id", "quant_dir_name", "quant_layout", "lib_format_json_path"
        ).iter_rows()
    ],
    how="vertical_relaxed",
).select(
    "sample_id",
    "quant_dir_name",
    "quant_layout",
    "lib_format_json_path",
    pl.exclude("sample_id", "quant_dir_name", "quant_layout", "lib_format_json_path"),
)

# Read each matched JSON (one row per file) and concatenate
lib_format_counts = pl.concat(
    [
        pl.read_json(path).with_columns(
            pl.lit(sample_id).alias("sample_id"),
            pl.lit(quant_dir_name).alias("quant_dir_name"),
            pl.lit(quant_layout).alias("quant_layout"),
            pl.lit(path).alias("lib_format_json_path"),
        )
        for sample_id, quant_dir_name, quant_layout, path in found_lib_format.select(
            "sample_id", "quant_dir_name", "quant_layout", "lib_format_json_path"
        ).iter_rows()
    ],
    how="vertical_relaxed",
).select(
    "sample_id",
    "quant_dir_name",
    "quant_layout",
    "lib_format_json_path",
    pl.exclude("sample_id", "quant_dir_name", "quant_layout", "lib_format_json_path"),
)

pl.Config.set_tbl_rows(2)
print(f"lib_format_counts rows: {lib_format_counts.height}")
display(lib_format_counts)

pl.Config.set_tbl_rows(20)
display(
    lib_format_counts.group_by("quant_layout", "expected_format")
    .len()
    .sort("quant_layout", "expected_format")
)

lib_format_counts rows: 744


sample_id,quant_dir_name,quant_layout,lib_format_json_path,read_files,expected_format,compatible_fragment_ratio,num_compatible_fragments,num_assigned_fragments,num_frags_with_concordant_consistent_mappings,num_frags_with_inconsistent_or_orphan_mappings,strand_mapping_bias,MSF,OSF,ISF,MSR,OSR,ISR,SF,SR,MU,OU,IU,U
str,str,str,str,str,str,f64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""SRR27681984""","""SRR27681984""","""HS_human_salmon_quant_se_gencode""","""../../out/salmon_gencode_result/HS_human_salmon_quant_se_gencode/SRR27681984/lib_format_counts.json""","""[ /var/lib/cwl/stga30ec5b0-5093-47bf-b485-dce08416203c/SRR27681984_trim.fq.gz ]""","""SR""",1.0,31523061,31523061,31523049,152,0.000005,0,0,0,0,0,0,152,31523049,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HRR255912""","""HRR255912_1""","""HS_human_salmon_quant_pe_gencode""","""../../out/salmon_gencode_result/HS_human_salmon_quant_pe_gencode/HRR255912_1/lib_format_counts.json""","""[ /var/lib/cwl/stg24f5be00-21c7-4f90-8219-b56df24a217b/HRR255912_1_trim.fq.gz, /var/lib/cwl/stgec715520-a99f-4ebc-9b0b-638eae987a65/HRR255912_2_trim.fq.gz]""","""IU""",1.0,8562374,8562374,8666571,107450,0.502765,0,0,4357250,0,0,4309321,47472,59978,0,0,0,0


quant_layout,expected_format,len
str,str,u32
"""HS_human_salmon_quant_pe_gencode""","""ISF""",48
"""HS_human_salmon_quant_pe_gencode""","""ISR""",287
"""HS_human_salmon_quant_pe_gencode""","""IU""",192
"""HS_human_salmon_quant_pe_gencode""","""SR""",2
"""HS_human_salmon_quant_se_gencode""","""SF""",28
"""HS_human_salmon_quant_se_gencode""","""SR""",159
"""HS_human_salmon_quant_se_gencode""","""U""",28


&nbsp;

&nbsp;

## Heat / control pair join

- `rnaseq_all_metadata` から `sample_heat_id` と `sample_control_id` だけを抜き出す（406 ペア）
- `lib_format_counts` の `expected_format` と `strand_mapping_bias` だけを heat 側・control 側それぞれに left join する
- salmon の [LIBTYPE](https://salmon.readthedocs.io/en/latest/salmon.html#what-s-this-libtype) に従い、heat / control の両方が `S` なら stranded、両方が `U` なら unstranded とラベルする

In [6]:
# 405 heat/control pairs; join lib_format_counts onto each ID
hn_pairs = rnaseq_all_metadata.select("sample_heat_id", "sample_control_id")

lib_format_heat = lib_format_counts.select(
    pl.col("sample_id"),
    pl.col("expected_format", "strand_mapping_bias").name.suffix("_heat"),
)
lib_format_control = lib_format_counts.select(
    pl.col("sample_id"),
    pl.col("expected_format", "strand_mapping_bias").name.suffix("_control"),
)

hn_lib_format = hn_pairs.join(
    lib_format_heat,
    left_on="sample_heat_id",
    right_on="sample_id",
    how="left",
    maintain_order="left",
    validate="m:1",
).join(
    lib_format_control,
    left_on="sample_control_id",
    right_on="sample_id",
    how="left",
    maintain_order="left",
    validate="m:1",
).with_columns(
    pl.when(
        pl.col("expected_format_heat") == pl.col("expected_format_control")
    ).then(
        pl.lit("same format")
    ).otherwise(
        pl.lit("different format")
    ).alias("format_pair_comparison"),

    # validate strandedness
    # Salmon LIBTYPE: S = stranded, U = unstranded (https://salmon.readthedocs.io/en/latest/salmon.html#what-s-this-libtype)
    pl.when(
        pl.col("expected_format_heat").str.contains("S", literal=True),
        pl.col("expected_format_control").str.contains("S", literal=True),
    )
    .then(pl.lit("stranded library pairs"))
    .when(
        pl.col("expected_format_heat").str.contains("U", literal=True),
        pl.col("expected_format_control").str.contains("U", literal=True),
    )
    .then(pl.lit("unstranded library pairs"))
    .otherwise(pl.lit("mixed strandedness library pairs"))
    .alias("expected_library_strandedness"),
)

pl.Config.set_tbl_rows(2)
print(f"hn_lib_format rows: {hn_lib_format.height}")
display(hn_lib_format)

pl.Config.set_tbl_rows(20)
display(
    hn_lib_format.select(
        "expected_format_heat",
        "expected_format_control",
        "format_pair_comparison",
        "expected_library_strandedness",
    )
    .group_by(
        "expected_format_heat",
        "expected_format_control",
        "format_pair_comparison",
        "expected_library_strandedness",
    )
    .len()
    .sort("len", descending=True)
)


display(
    hn_lib_format.select(
        "expected_library_strandedness"
    )
    .group_by("expected_library_strandedness")
    .len()
    .sort("len", descending=True)
)

hn_lib_format rows: 406


sample_heat_id,sample_control_id,expected_format_heat,strand_mapping_bias_heat,expected_format_control,strand_mapping_bias_control,format_pair_comparison,expected_library_strandedness
str,str,str,f64,str,f64,str,str
"""SRR27681984""","""SRR27681986""","""SR""",0.000005,"""SR""",0.000004,"""same format""","""stranded library pairs"""
…,…,…,…,…,…,…,…
"""HRR255914""","""HRR255912""","""IU""",0.502674,"""IU""",0.502765,"""same format""","""unstranded library pairs"""


expected_format_heat,expected_format_control,format_pair_comparison,expected_library_strandedness,len
str,str,str,str,u32
"""ISR""","""ISR""","""same format""","""stranded library pairs""",146
"""IU""","""IU""","""same format""","""unstranded library pairs""",111
"""SR""","""SR""","""same format""","""stranded library pairs""",85
"""ISF""","""ISF""","""same format""","""stranded library pairs""",28
"""U""","""U""","""same format""","""unstranded library pairs""",15
"""SF""","""SF""","""same format""","""stranded library pairs""",14
"""ISR""","""IU""","""different format""","""mixed strandedness library pairs""",5
"""U""","""SR""","""different format""","""mixed strandedness library pairs""",2


expected_library_strandedness,len
str,u32
"""stranded library pairs""",273
"""unstranded library pairs""",126
"""mixed strandedness library pairs""",7


&nbsp;

&nbsp;

&nbsp;

## Pie Chart

In [7]:
# Colors for salmon LIBTYPE strandedness of heat/control pairs
library_strandedness_colors = {
    "stranded library pairs": "#EF553B",
    "unstranded library pairs": "#636EFA",
    "mixed strandedness library pairs": "#D3D3D3",
}

library_strandedness_counts = (
    hn_lib_format
    .group_by("expected_library_strandedness")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

fig_basic = px.pie(
    library_strandedness_counts.to_pandas(),
    values="count",
    names="expected_library_strandedness",
    title="Expected library strandedness (heat / control pairs)",
    color="expected_library_strandedness",
    color_discrete_map=library_strandedness_colors,
)

fig_basic.update_traces(
    textposition="outside",
    textinfo="value+percent+label",
    hovertemplate="<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>",
)

fig_basic.update_layout(
    font_size=18,
    showlegend=True,
    legend={
        "orientation": "v",
        "yanchor": "middle",
        "y": 0.5,
        "xanchor": "left",
        "x": 1.01,
    },
)

fig_basic.show()

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 5. Join Dataframe

- `rnaseq_all_metadata` と `hn_lib_format` を `join` する

In [8]:
rnaseq_all_metadata_all_columns = pl.read_csv(
    "../../Data/metadata/metadata_2610/rna_seq_all_metadata_all_columns_261001.tsv",
    separator="\t",
    schema_overrides={
        "sample_contact_phone": pl.Utf8
    }
)

In [10]:
rnaseq_all_metadata_join_salmon = rnaseq_all_metadata.join(
    hn_lib_format,
    on=["sample_heat_id", "sample_control_id"],
    how="inner",
    nulls_equal=True,
    validate="1:1"
)

rnaseq_all_metadata_join_salmon.write_csv(
    "../../Data/metadata/metadata_2610/rna_seq_all_metadata_simple_join_salmon_261001.tsv",
    separator="\t"
)

display(rnaseq_all_metadata_join_salmon)

sample_heat_id,sample_control_id,sample_name,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,sample,stress treatment temperature,stress treatment time,control treatment temperature,rna_library,normalized_label,item_type,reference_doi,strandness,strandness_evidence,expected_format_heat,strand_mapping_bias_heat,expected_format_control,strand_mapping_bias_control,format_pair_comparison,expected_library_strandedness
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,f64,str,str
"""SRR27681984""","""SRR27681986""","""wi-38-htert_40°c_rna-seq_rep1 vs wi-38-htert_37°c_rna-seq_rep1""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 37°C, RNA-seq, rep1""","""SRX23349232 vs SRX23349230""","""GSM8029688_r1 vs GSM8029686_r1""","""SRS20210616 vs SRS20210614""","""PRJNA1067599""","""2024-10-16""","""GSE253874""","""ILLUMINA""","""Illumina HiSeq 2000""","""TRANSCRIPTOMIC""","""SINGLE""","""total RNA""","""WI-38-hTERT""","""40 C""","""120 min.""","""37 C""","""TruSeq Stranded Total RNA Library Prep Kit""","""TruSeq Stranded Total RNA Library Prep Kit""","""library_preparation""","""https://doi.org/10.1111/gtc.13140""","""stranded""","""https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-refer…","""SR""",0.000005,"""SR""",0.000004,"""same format""","""stranded library pairs"""
"""SRR27681983""","""SRR27681985""","""wi-38-htert_40°c_rna-seq_rep2 vs wi-38-htert_37°c_rna-seq_rep2""","""WI-38-hTERT, 40°C, RNA-seq, rep2 vs WI-38-hTERT, 37°C, RNA-seq, rep2""","""SRX23349233 vs SRX23349231""","""GSM8029689_r1 vs GSM8029687_r1""","""SRS20210615 vs SRS20210617""","""PRJNA1067599""","""2024-10-16""","""GSE253874""","""ILLUMINA""","""Illumina HiSeq 2000""","""TRANSCRIPTOMIC""","""SINGLE""","""total RNA""","""WI-38-hTERT""","""40 C""","""120 min.""","""37 C""","""TruSeq Stranded Total RNA Library Prep Kit""","""TruSeq Stranded Total RNA Library Prep Kit""","""library_preparation""","""https://doi.org/10.1111/gtc.13140""","""stranded""","""https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-refer…","""SR""",0.000006,"""SR""",0.000006,"""same format""","""stranded library pairs"""
"""SRR24973772""","""SRR24973776""","""m3d_heat_1_rna_seq vs m3d_ctrl_1_rna_seq""","""M3D_Heat_1_RNA_seq vs M3D_Ctrl_1_RNA_seq""","""SRX20731424 vs SRX20731421""","""GSM7500881_r1 vs GSM7500878_r1""","""SRS18022362 vs SRS18022360""","""PRJNA985730""","""2023-06-27""","""GSE235351""","""BGISEQ""","""MGISEQ-2000RS""","""TRANSCRIPTOMIC""","""SINGLE""","""total RNA""","""MHCC-97H""","""37~53 C""","""15 min.""","""37 C (inferred)""","""unknown""","""unknown""","""missing""","""https://doi.org/10.1016/j.xcrm.2023.101128""","""unknown""","""""","""U""",0.501465,"""U""",0.499687,"""same format""","""unstranded library pairs"""
"""SRR24973774""","""SRR24973775""","""m3d_heat_2_rna_seq vs m3d_ctrl_2_rna_seq""","""M3D_Heat_2_RNA_seq vs M3D_Ctrl_2_RNA_seq""","""SRX20731425 vs SRX20731422""","""GSM7500882_r1 vs GSM7500879_r1""","""SRS18022364 vs SRS18022361""","""PRJNA985730""","""2023-06-27""","""GSE235351""","""BGISEQ""","""MGISEQ-2000RS""","""TRANSCRIPTOMIC""","""SINGLE""","""total RNA""","""MHCC-97H""","""37~53 C""","""15 min.""","""37 C (inferred)""","""unknown""","""unknown""","""missing""","""https://doi.org/10.1016/j.xcrm.2023.101128""","""unknown""","""""","""U""",0.499229,"""U""",0.497806,"""same format""","""unstranded library pairs"""
"""SRR24973771""","""SRR24973773""","""m3d_heat_3_rna_seq vs m3d_ctrl_3_rna_seq""","""M3D_Heat_3_RNA_seq vs M3D_Ctrl_3_RNA_seq""","""SRX20731426 vs SR

In [11]:
rnaseq_all_metadata_join_salmon_all_columns = rnaseq_all_metadata_join_salmon.join(
    rnaseq_all_metadata_all_columns,
    on=rnaseq_all_metadata.columns,
    how="inner",
    nulls_equal=True,
    validate="1:1"
)

rnaseq_all_metadata_join_salmon_all_columns.write_csv(
    "../../Data/metadata/metadata_2610/rna_seq_all_metadata_simple_join_salmon_all_columns_261001.tsv",
    separator="\t"
)

display(rnaseq_all_metadata_join_salmon_all_columns)

sample_heat_id,sample_control_id,sample_name,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,sample,stress treatment temperature,stress treatment time,control treatment temperature,rna_library,normalized_label,item_type,reference_doi,strandness,strandness_evidence,expected_format_heat,strand_mapping_bias_heat,expected_format_control,strand_mapping_bias_control,format_pair_comparison,expected_library_strandedness,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,…,scientific_name,tax_id,accession,collection_date_end,collection_date_start,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,nominal_sdev,broker_name,datahub,nominal_length,submitted_read_type,submitted_format,bam_ftp,bam_bytes,dev_stage,bam_aspera,bam_galaxy,bam_file_role,submitted_file_role,recovery treatment temperature,recovery treatment time,sample_series_id,biosample,sra,fastq_galaxy,fastq_ftp,fastq_aspera,tax_lineage,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,f64,str,str,str,str,str,str,str,i64,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR27681984""","""SRR27681986""","""wi-38-htert_40°c_rna-seq_rep1 vs wi-38-htert_37°c_rna-seq_rep1""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 37°C, RNA-seq, rep1""","""SRX23349232 vs SRX23349230""","""GSM8029688_r1 vs GSM8029686_r1""","""SRS20210616 vs SRS20210614""","""PRJNA1067599""","""2024-10-16""","""GSE253874""","""ILLUMINA""","""Illumina HiSeq 2000""","""TRANSCRIPTOMIC""","""SINGLE""","""total RNA""","""WI-38-hTERT""","""40 C""","""120 min.""","""37 C""","""TruSeq Stranded Total RNA Library Prep Kit""","""TruSeq Stranded Total RNA Library Prep Kit""","""library_preparation""","""https://doi.org/10.1111/gtc.13140""","""stranded""","""https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-refer…","""SR""",0.000005,"""SR""",0.000004,"""same format""","""stranded library pairs""","""GSM8029688 vs GSM8029686""","""Public on Oct 15 2024""","""Jan 22 2024""","""Oct 15 2024""","""SRA""",1,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""SRR27681983""","""SRR27681985""","""wi-38-htert_40°c_rna-seq_rep2 vs wi-38-htert_37°c_rna-seq_rep2""","""WI-38-hTERT, 40°C, RNA-seq, rep2 vs WI-38-hTERT, 37°C, RNA-seq, rep2""","""SRX23349233 vs SRX23349231""","""GSM8029689_r1 vs GSM8029687_r1""","""SRS20210615 vs SRS20210617""","""PRJNA1067599""","""2024-10-16""","""GSE253874""","""ILLUMINA""","""Illumina HiSeq 2000""","""TRANSCRIPTOMIC""","""SINGLE""","""total RNA""","""WI-38-hTERT""","""40 C""","""120 min.""","""37 C""","""TruSeq Stranded Total RNA Library Prep Kit""","""TruSeq Stranded Total RNA Library Prep Kit""","""library_preparation""","""https://doi.org/10.1111/gtc.13140""","""stranded""","""https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-refer…","""SR""",0.000006,"""SR""",0.000006,"""same format""","""stranded library pairs""","""GSM8029689 vs GSM8029687""","""Public on Oct 15 2024""","""Jan 22 2024""","""Oct 15 2024""","""SRA""",1,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null